# Readers

Notebook was prepared as a sample tool to read data from different data sources. Moreover notebook includes some simple exploratory data analysis for data overview purposes.

In [ ]:
import os
import sys
from pathlib import Path
from pprint import pprint as print

import requests
import pandas as pd
import numpy as np


# Setup pitch and plot
from mplsoccer import Pitch, VerticalPitch


In [ ]:
CURRENT_WORKING_DIR = Path(os.getcwd())
WORKING_DIR = CURRENT_WORKING_DIR.parent
DATA_DIR = WORKING_DIR / "data"

sys.path.append(str(WORKING_DIR))

## SkillCorner

Data loading is based on the SkillCorner tutorial guide. More information under below link :arrow_down:

**Link:** https://github.com/SkillCorner/opendata

In [ ]:
# Content URL for the SkillCorner opendata repository
content_url = "https://api.github.com/repos/SkillCorner/opendata/contents/data/matches"

# Get the list of available matches
response = requests.get(content_url)
matches = response.json()

# Display available match IDs
match_ids = [match['name'] for match in matches if match['type'] == 'dir']
print(f"Available matches: {len(match_ids)}")
print(match_ids)

## Data Readers ###

In [ ]:
# Example: Load tracking data for a specific match
match_id = 1886347

# Construct the raw tracking content URL
content_tracking_url = content_url + f"/{match_id}/{match_id}_tracking_extrapolated.jsonl"  # Data is stored using GitLFS
content_tracking_data = pd.read_json(content_tracking_url, lines=True)
print(f"Downloaded tracking data URL: {content_tracking_data['download_url'][0]}")

# Construct the raw match content URL
content_match_url = content_url + f"/{match_id}/{match_id}_match.json"  # Data is stored using GitLFS
content_match_data = pd.read_json(content_match_url, lines=True)
print(f"Downloaded match data URL: {content_match_data['download_url'][0]}")

## Player data

Extract player and ball data. Preprocess raw data frame into processed, ready for analytics data frame. 

In [ ]:
from football_ml.jobs.data_gathering_job import preprocess_tracking_data

# Read the JSON tracking data as a JSON object
raw_tracking_data = pd.read_json(content_tracking_data['download_url'][0], lines=True)

# Process the raw data
processed_tracking_df = preprocess_tracking_data(raw_tracking_data)
processed_tracking_df["match_id"] = match_id
processed_tracking_df.head()

In [ ]:
path = DATA_DIR / "raw" / "all_matches_tracking.parquet"
all_matches_tracking_df = pd.read_parquet(path)

## Match data

In [ ]:
from football_ml.jobs.data_gathering_job import preprocess_players_data

# Read the JSON match data as a JSON object
response = requests.get(content_match_data['download_url'][0])
raw_match_data = response.json()
raw_match_df = pd.json_normalize(raw_match_data, max_level=2)
processed_players_df = preprocess_players_data(raw_match_df)
processed_players_df.head()

In [ ]:
path = DATA_DIR / "raw" / "all_matches_players.parquet"
all_matches_players_df = pd.read_parquet(path)

## Merging data frame

In [ ]:
merged_df = all_matches_tracking_df.merge(
    all_matches_players_df, on=["player_id", "match_id"], how="left"
)
merged_df["time_in_minutes"] = np.round(merged_df["timestamp"].dt.hour * 60 + merged_df["timestamp"].dt.minute + merged_df["timestamp"].dt.second / 60, 2)
merged_df.head()

In [ ]:
print(f"Processed tracking data shape: {all_matches_tracking_df.shape}")
print(f"Processed players data shape: {all_matches_players_df.shape}")
print(f"Merged data shape: {merged_df.shape}")

**Analysis ideas:**
 - free kicks, corners - players movements
 - Midfielders - off ball movements, number of offensive / defensive passes, passes based on the number of opponent players
 - Center Backs - off ball movements in offensive part of the game, passes
 - Before goal team movement

**ML model ideas:**
- Clustering - players similarities
    - off ball movements

In [ ]:
# Filtering for frames with a team in possession
filtered_df = merged_df[
    merged_df["possession_group"].notnull()
].copy()


# We basically want to convert the X and Y to make sure we're always visualizing left to right. A player's XY will depend on the half so the straight average doesn't work

filtered_df["direction_player"] = np.where(
    filtered_df["period"] == 1,
    filtered_df["direction_player_1st_half"],
    filtered_df["direction_player_2nd_half"],
)
filtered_df["x"] = np.where(
    filtered_df["direction_player"] == "right_to_left",
    -filtered_df["x"],
    filtered_df["x"],
)  # Convert X
filtered_df["y"] = np.where(
    filtered_df["direction_player"] == "right_to_left",
    -filtered_df["y"],
    filtered_df["y"],
)  # Convert Y

# Create some flags in case we need them later
filtered_df["possession_team_name"] = np.where(
    filtered_df["possession_group"] == "home team",
    filtered_df["home_team.name"],
    filtered_df["away_team.name"],
)
filtered_df["possession_flag"] = np.where(
    filtered_df["possession_team_name"] == filtered_df["team_name"], "IP", "OOP"
)


# At this point all of our players have their X and Y adjusted from left to right , so we can aggregate for across the game

aggregated_df = (
    filtered_df.groupby(
        [
            "player_id",
            "possession_group",
            "team_name",
            "possession_team_name",
            "possession_flag",  # This will allow us to check the position of the player when the team is in or out of possession
            "start_time",  # With this we can filter in or out the players that came on as subs
            "number",
            "is_gk",
        ]
    )[["x", "y"]]
    .mean()
    .reset_index()
)

aggregated_df.sort_values(["possession_flag", "team_name"])

# Visualization example

In [ ]:
mask = (merged_df['player_id'] == 23909) & (merged_df['match_id'] == 1899585) & ()
viz_df = merged_df[mask]

In [ ]:
merged_df.columns

In [ ]:
!pip install nbformat==4.3.0

In [ ]:
from plotly import graph_objects as go

mask = viz_df["period"] == 1


In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=viz_df.loc[mask, 'time_in_minutes'],
    y=viz_df.loc[mask, 'x'],
    mode='lines+markers',
    name='x'
))
fig.add_trace(go.Scatter(
    x=viz_df.loc[mask, 'time_in_minutes'],
    y=viz_df.loc[mask, 'ball_x'],
    mode='lines+markers',
    name='ball_x'
))
fig.update_layout(
    title="Player X Position and Ball X Position Over Time (Period 1)",
    xaxis_title="Time (minutes)",
    yaxis_title="X Position (meters)",
    legend_title="Legend"
)
fig.show()

In [ ]:

# plt.scatter(viz_df.loc[mask, 'time_in_minutes'], viz_df.loc[mask, 'x'], s=10, marker='.', label='x')
plt.plot(viz_df.loc[mask, 'time_in_minutes'], viz_df.loc[mask, 'x'], label='x')
# plt.scatter(viz_df.loc[mask, 'time_in_minutes'], viz_df.loc[mask, 'ball_x'], s=10, marker='.', label='ball_x')
plt.plot(viz_df.loc[mask, 'time_in_minutes'], viz_df.loc[mask, 'ball_x'], label='ball_x')
plt.legend()

In [ ]:
# Teams
aggregated_df.team_name.unique().tolist()

In [ ]:
possession = "IP"  # 'IP' or 'OOP'
team = 'Brisbane Roar FC'  # Pick one team, you can use the name directly

In [ ]:
pitch = Pitch(
    pitch_type="skillcorner",
    line_alpha=0.75,
    pitch_length=105,
    pitch_width=68,
    pitch_color="#001400",
    line_color="white",
    linewidth=1.5,
)
fig, ax = pitch.grid(figheight=8, endnote_height=0, title_height=0)

viz_ip = aggregated_df[
    (aggregated_df["possession_flag"] == possession)
    & (aggregated_df["team_name"] == team)
].reset_index(drop=True)

ax.scatter(
    viz_ip["x"],
    viz_ip["y"],
    c="#32FE6B",
    alpha=0.95,
    s=600,
    edgecolors="white",
    linewidths=2.5,
    zorder=10,
    label="team",
)

# Annotate player numbers
for i, row in viz_ip.iterrows():
    ax.text(
        row["x"],
        row["y"],
        str(row["number"]),
        color="black",
        fontweight="bold",
        fontsize=10,
        ha="center",
        va="center",
        zorder=16,
    )

ax.set_title(f"{team} Average Positions in Possession")

In [ ]:
aggregated_df["start_time"].unique()

In [ ]:
pitch = Pitch(
    pitch_type="skillcorner",
    line_alpha=0.75,
    pitch_length=105,
    pitch_width=68,
    pitch_color="#001400",
    line_color="white",
    linewidth=1.5,
)
fig, ax = pitch.grid(figheight=8, endnote_height=0, title_height=0)

viz_ip = aggregated_df[
    (aggregated_df["possession_flag"] == possession)
    & (aggregated_df["team_name"] == team)
    & (aggregated_df["start_time"] == "00:00:00")
].reset_index(drop=True)


ax.scatter(
    viz_ip["x"],
    viz_ip["y"],
    c="#32FE6B",
    alpha=0.95,
    s=600,
    edgecolors="white",
    linewidths=2.5,
    zorder=10,
    label="team",
)


# Annotate player numbers
for i, row in viz_ip.iterrows():
    ax.text(
        row["x"],
        row["y"],
        str(row["number"]),
        color="black",
        fontweight="bold",
        fontsize=10,
        ha="center",
        va="center",
        zorder=16,
    )

ax.set_title(f"{team} Average Positions in Possession")

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

import matplotlib.pyplot as plt

# Create pitch
pitch = Pitch(
    pitch_type="skillcorner",
    line_alpha=0.75,
    pitch_length=105,
    pitch_width=68,
    pitch_color="#001400",
    line_color="white",
    linewidth=1.5,
)

# Select a player to visualize
player_id = 23909  # You can change this to any player_id from the tracking data
player_name = merged_df[merged_df['player_id'] == player_id]['short_name'].values[0]
player_number = merged_df[merged_df['player_id'] == player_id]['number'].values[0]

# Filter tracking data for the selected player
player_tracking = merged_df[
    merged_df['player_id'] == player_id
].copy()

# Adjust coordinates based on direction (similar to what was done in filtered_df)
player_tracking["direction_player"] = np.where(
    player_tracking["period"] == 1,
    player_tracking["direction_player_1st_half"],
    player_tracking["direction_player_2nd_half"],
)
player_tracking["x_adjusted"] = np.where(
    player_tracking["direction_player"] == "right_to_left",
    -player_tracking["x"],
    player_tracking["x"],
)
player_tracking["y_adjusted"] = np.where(
    player_tracking["direction_player"] == "right_to_left",
    -player_tracking["y"],
    player_tracking["y"],
)

# Create figure
fig, ax = pitch.draw(figsize=(12, 8))

# Create heatmap using hexbin
heatmap = ax.hexbin(
    player_tracking['x_adjusted'],
    player_tracking['y_adjusted'],
    gridsize=20,
    cmap='hot',
    alpha=0.7,
    edgecolors='none',
    mincnt=1
)

# Add colorbar
cbar = plt.colorbar(heatmap, ax=ax)
cbar.set_label('Frequency', rotation=270, labelpad=20)

# Add title
plt.title(
    f"Heat Map: {player_name} (#{player_number}) - Full Match Movement",
    fontsize=14,
    fontweight='bold',
    color='white'
)

plt.tight_layout()
fig.show()